In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd

# Imports data from module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"
host = 'localhost'
port = 27017
database = 'AAC'
collection = 'animals'

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password, host, port, database, collection)

# Class read method must support return of list object
# and accept projection JSON input.
# Sending the read method an empty document requests
# all documents be returned.
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB returns the '_id' column as an ObjectId.
# Dash DataTable cannot properly display ObjectId values,
# so the column is removed from the DataFrame.
df.drop(columns=['_id'], inplace=True)


## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)


app.layout = html.Div([

    html.Div(id='hidden-div', style={'display': 'none'}),
    dcc.Store(id='refresh-data',data=0),
    html.H1('CS-340 Dashboard', style={'textAlign': 'center'}),
    html.Div(id="query-out", style={'whiteSpace': 'pre-line'}),
    html.H1("Kaitlin's Client Server Authentication Dashboard:", style={'textAlign': 'center'}),

    html.Hr(),

    ################################
    # Rescue Type Filtering Options
    ################################

    html.Div([
        html.Label("Filter By Rescue Type:"),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness'},
                {'label': 'Disaster or Individual Rescue', 'value': 'Disaster or Individual'},
                {'label': 'Reset', 'value': 'All'},
            ],
            value='All',
            inline=True
        )

    ], style={'padding': '10px'}),

    html.Hr(),


    ############################
    # Animal Management Interface
    ############################

    html.Div([

        html.H2("Animal Management"),

        # animal name
        html.Div([
            html.Label("Animal Name:"),
            html.Br(),

            dcc.Input(id='animal-name', type='text', placeholder='Enter animal name', style={'width': '300px'})              
        ], style={'marginBottom': '10px'}),


        # animal type
        html.Div([
            html.Label("Animal Type:"),

            dcc.Dropdown(
                id='animal-type',
                options=[
                    {'label': 'Dog','value': 'Dog'},
                    {'label': 'Cat','value': 'Cat'},
                    {'label': 'Other','value': 'Other'}
                ],
                placeholder='Select animal type', value=None, clearable=True,
                style={'width': '300px'}
            )

        ], style={'width': '300px', 'marginBottom': '10px'}),


        # animal breed
        html.Div([
            html.Label("Animal Breed:"),
            html.Br(),

            dcc.Input(id='animal-breed', type='text', placeholder='Enter animal breed', style={'width': '300px'})

        ], style={'marginBottom': '10px'}),


        # animal age
        html.Div([
            html.Label("Animal Age (in weeks):"),
            html.Br(),

            dcc.Input(id='animal-age', type='number', min=0, placeholder='Enter animal age', style={'width': '300px'})

        ], style={'marginBottom': '10px'}),


        # animal sex
        html.Div([
            html.Label("Animal Sex Upon Outcome:"),

            dcc.Dropdown(
                id='animal-sex',
                options=[
                    {'label': 'Intact Male', 'value': 'Intact Male'},
                    {'label': 'Intact Female','value': 'Intact Female'},
                    {'label': 'Neutered Male','value': 'Neutered Male'},
                    {'label': 'Spayed Female','value': 'Spayed Female'},
                    {'label': 'Unknown','value': 'Unknown'}
                ],
                placeholder='Select animal sex', value=None, clearable=True,
                style={'width': '300px'}
            )

        ], style={'width': '300px', 'marginBottom': '15px'}),

        ########################
        # Management Buttons
        ########################

        html.Button(
            "Add Animal",
            id="add-animal-btn",
            n_clicks=0
        ),

        html.Button(
            "Update Animal",
            id="update-animal-btn",
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),

        html.Button(
            "Delete Animal",
            id="delete-animal-btn",
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),

        html.Button(
            "Clear Form",
            id="clear-animal-btn",
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),


        # Status Message
        html.Div(
            id='animal-management-status',
            style={'marginTop': '10px'}
        )

    ], style={
        'padding': '15px',
        'border': '1px solid #ccc',
        'marginBottom': '20px'
    }),


    ########################
    # Interactive DataTable
    ########################

    dash_table.DataTable(
       id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=True,                                #allows user to edit cells
        filter_action="native",                       #Column filtering
        sort_action="native",                         #sort by column
        sort_mode="multi",                            #sort multiple columns
        row_selectable="single",                      #allows user to select a row
        selected_rows=[0],                            #tracks selected row(s)
        row_deletable=False,                          #Prevents user from deleting row(s)
        column_selectable="single",                   #allows user to select a column
        selected_columns=[],                          #tracks selected column(s)
        page_action="native",                         #built-in pagination, 10 rows/pg
        page_current=0,
        page_size=10,
    ),

    html.Br(),
    html.Hr(),


    ########################
    # Graph and Map
    ########################

    # Sets up the dashboard so that the chart
    # and geolocation map are side-by-side.
    html.Div(className='row', style={'display': 'flex'}, children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ])
])


#############################################
# Interaction Between Components / Controller
#############################################


########################
# Rescue Type Filtering
########################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value'),
     Input('refresh-data','data')]
)

def update_dashboard(filter_type, refresh_data):
    query = {}

    if filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26,"$lte": 156},
            "breed": {"$in": ["Labrador Retriever Mix","Chesa Bay Retr Mix","Newfoundland Mix"]}
        }

    elif filter_type == 'Mountain or Wilderness':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26,"$lte": 156},
            "breed": {"$in": ["German Shepherd","Alaskan Malamute","Old English Sheepdog","Siberian Husky","Rottweiler"]}
        }

    elif filter_type == 'Disaster or Individual':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20,"$lte": 300 },
            "breed": {"$in": ["Doberman Pinscher","German Shepherd","Golden Retriever","Bloodhound","Rottweiler"]}
        }

    else:
        query = {}

    return (
        pd.DataFrame.from_records(shelter.read(query)).drop(columns=['_id'], errors = 'ignore').to_dict('records'))


########################
# Pie Chart
########################

# Display the breeds of animals based on quantity
# represented in the DataTable.

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)

def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    df_view = pd.DataFrame(viewData)
    fig = px.pie(df_view,names='breed',title='Preferred Animals')

    return [dcc.Graph(figure=fig)]


########################
# Selected Column Styling
########################

# Highlights a column when the user selects it.

@app.callback(
    Output('datatable-id','style_data_conditional'),
    [Input('datatable-id','selected_columns')]
)

def update_styles(selected_columns):

    return [ {
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        } for i in selected_columns]

#########################################
# Selected animal management & clear form
##########################################

#populates animal management form with selected row data
@app.callback([
    Output('animal-name', 'value'),
    Output('animal-type', 'value'),
    Output('animal-breed', 'value'),
    Output('animal-age', 'value'),
    Output('animal-sex', 'value')
],
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows'),
     Input('clear-animal-btn', 'n_clicks')]
)

def populate_animal_management_form(viewData, selected_rows, clear_clicks):

    #determine what causes the callback
    triggered_id = dash.ctx.triggered_id

    #clear all form fields when clear form button is clicked
    if triggered_id == 'clear-animal-btn':
        return [None, None, None, None, None]

    # If no row is selected, return None for all form fields.
    if not viewData:
        return [None, None, None, None, None]

    #use selected row or default to first row if no selection
    row_index = selected_rows[0] if selected_rows else 0

    #make sure the selectec row exists
    if row_index >= len(viewData):
        return [None, None, None, None, None]

    animal = viewData[row_index]

    return (
        animal.get('name'),
        animal.get('animal_type'),
        animal.get('breed'),
        animal.get('age_upon_outcome_in_weeks'),
        animal.get('sex_upon_outcome')
    )

########################
# Add Animal
########################

#adds new animal record to DB

@app.callback(
    [Output('animal-management-status', 'children'),
    Output('refresh-data','data')],
    Input('add-animal-btn', 'n_clicks'),
    [State('animal-name', 'value'),
     State('animal-type', 'value'),
     State('animal-breed', 'value'),
     State('animal-age', 'value'),
     State('animal-sex', 'value'),
     State('refresh-data', 'data')],
     prevent_initial_call=True
)

def add_animal(n_clicks, name, animal_type, breed, age, sex, refresh_data):

    #validate required fields before adding animal
    if n_clicks > 0:
        if not all([name, animal_type, breed, age, sex]):
            return "Please fill in all fields before adding an animal.", refresh_data

        #create a new animal record
        new_animal = {
            "name": name.strip(),
            "animal_type": animal_type,
            "breed": breed,
            "age_upon_outcome_in_weeks": age,
            "sex_upon_outcome": sex
        }

        #add the new animal to the database successfully
        success = shelter.create(new_animal)

        if success:
            return f"Animal '{name}' added successfully!", refresh_data + 1

        return "Error: Failed to add animal. Please try again.", refresh_data

########################
# Update Animal
########################

#updates selected animal record in DB

@app.callback(
    [Output('animal-management-status', 'children', allow_duplicate=True),
    Output('refresh-data','data', allow_duplicate=True)],
    Input('update-animal-btn', 'n_clicks'),

    [State('datatable-id', 'derived_virtual_data'),
     State('datatable-id', 'derived_virtual_selected_rows'),
     State('animal-name', 'value'),
     State('animal-type', 'value'),
     State('animal-breed', 'value'),
     State('animal-age', 'value'),
     State('animal-sex', 'value'),
     State('refresh-data', 'data')],

    prevent_initial_call=True)

def update_animal(n_clicks, viewData, selected_rows, name, animal_type, breed, age, sex, refresh_data):

    #ensure a record is selected
    if not viewData or not selected_rows:
        return "Please select an animal record to update.", refresh_data

    #validate required fields before updating animal
    if not name or not animal_type or not breed or age is None or not sex:
        return "Please fill in all fields before updating a record.", refresh_data

    #get the selected animal record
    row_index = selected_rows[0]

    if row_index >= len(viewData):
        return "Error: Selected record could not be found.", refresh_data

    selected_animal = viewData[row_index]

    #get the record number of the selected animal
    rec_num = selected_animal.get('rec_num')

    if rec_num is None:
        return "Error: No record number could be found.", refresh_data

    #updated values
    updated_values = {
        "name": name.strip(),
        "animal_type": animal_type,
        "breed": breed.strip(),
        "age_upon_outcome_in_weeks": age,
        "sex_upon_outcome": sex
    }

    #updaetd mongoDB
    modified_count = shelter.update({"rec_num": rec_num}, updated_values)

    if modified_count > 0:
        return f"Animal record '{name}' updated successfully!", refresh_data + 1

    return "No records were updated."

########################
# Delete Animal
########################

#Deletes selected animal record in DB

@app.callback(
    [Output('animal-management-status', 'children', allow_duplicate=True),
    Output('refresh-data','data', allow_duplicate=True)],
    Input('delete-animal-btn', 'n_clicks'),

    [State('datatable-id', 'derived_virtual_data'),
     State('datatable-id', 'derived_virtual_selected_rows'),
     State('refresh-data', 'data')],

    prevent_initial_call=True)

def delete_animal(n_clicks, viewData, selected_rows, refresh_data):

    #ensure a record is selected
    if not viewData or not selected_rows:
        return "Please select an animal record to delete.", refresh_data

    #get the selected animal record
    row_index = selected_rows[0]

    #make sure the selected row exists
    if row_index >= len(viewData):
        return "Error: Selected record could not be found.", refresh_data

    #get the selected animal record
    selected_animal = viewData[row_index]

    #get the record number of the selected animal
    rec_num = selected_animal.get('rec_num')

    if rec_num is None:
        return "Error: No record number could be found.", refresh_data

    #retrieve the name of the animal to be deleted for confirmation message
    animal_name = selected_animal.get('name', 'Unknown')

    #delete the animal record from the database
    deleted_count = shelter.delete({"rec_num": rec_num})

    if deleted_count > 0:
        return f"Animal record '{animal_name}' deleted successfully!", refresh_data + 1

    return "Error: Animal record could not be deleted. Please try again.", refresh_data

########################
# Geolocation Map
########################

# Updates the geolocation map for the selected
# animal in the DataTable.

@app.callback(
    Output('map-id','children'),
    [Input('datatable-id','derived_virtual_data'),
        Input('datatable-id','derived_virtual_selected_rows')]
)

def update_map(viewData, index):
    if not viewData or not index:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]

    #make sure selected row exists
    if row >= len(dff):
        return []

    #get lat & long 
    lattitude = dff.iloc[row].get('location_lat')
    longitude = dff.iloc[row].get('location_long')

    #if location data is missing, display message
    if pd.isna(lattitude) or pd.isna(longitude):
        return [html.Div([html.H3("Location unavailable."), 
                         html.P("No location data is available for this animal.")
                     ])
                ]

    #display map when valid coordinates available
    return [
        dl.Map(style={'width': '1000px','height': '500px'},center=[lattitude,longitude], zoom=10, children=[
            dl.TileLayer( id='base-layer-id'),
            dl.Marker(position=[lattitude,longitude],children=[
                dl.Tooltip(dff.iloc[row].get('name')),
                 dl.Popup([
                     html.H1( "Animal Name"),
                    html.P(dff.iloc[row].get('breed'))
                 ])
            ])
        ])
    ]



########################
# Run Application
########################

# Run app and display result.
# If port 8050 is already being used,
# an alternate port such as 8051 can be used.

app.run(debug=True)